# Catboost

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
df = pd.read_csv("../Day5/HR_comma_sep.csv")

In [3]:
df.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low


In [4]:
X,y = df.drop("left",axis=1), df["left"]
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=26,test_size=0.3,stratify=y)

In [5]:
ohc = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
ct = make_column_transformer((ohc,make_column_selector(dtype_include=object)),
                             ("passthrough", make_column_selector(dtype_include=['int64','float64'])),
                             verbose_feature_names_out=False)
ct = ct.set_output(transform='pandas')

In [9]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            cbc = CatBoostClassifier(random_state=26, verbose= False, learning_rate=l, n_estimators= n, max_depth= m)
            pipe = Pipeline([("CT",ct),("GDC",cbc)])
            pipe.fit(X_train,y_train)
            y_pred = pipe.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Score"])
df_scores.sort_values("Score",ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:38<00:00,  3.88s/it]


,Learning Rate,Max Depth,N,Score
98,0.801111,NaN,100,0.981551
110,0.900000,NaN,100,0.980440
86,0.702222,NaN,100,0.979773
119,0.900000,4.0,100,0.979106
74,0.603333,NaN,100,0.978884
...,...,...,...,...
5,0.010000,2.0,100,0.907090
7,0.010000,3.0,50,0.906646
6,0.010000,3.0,25,0.901534
4,0.010000,2.0,50,0.856635


In [11]:
df

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [12]:
categorical_features = ["Department","salary"]

In [15]:
cbc = CatBoostClassifier(random_state=26, verbose= False,cat_features=categorical_features, learning_rate=l, n_estimators= n, max_depth= m)
cbc.fit(X_train,y_train)
y_pred = cbc.predict(X_test)
accuracy_score(y_test,y_pred)

0.975994665481218

In [16]:
lr = np.linspace(0.01,0.9,10)
max_depth = [None, 2, 3, 4]
n_estimators = [25, 50, 100]
scores = []
for l in tqdm(lr):
    for m in max_depth:
        for n in n_estimators:
            cbc = CatBoostClassifier(random_state=26, cat_features=categorical_features,verbose= False, learning_rate=l, n_estimators= n, max_depth= m)
            cbc.fit(X_train,y_train)
            y_pred = cbc.predict(X_test)
            scores.append([l,m,n,accuracy_score(y_test, y_pred)])
df_scores = pd.DataFrame(scores, columns=["Learning Rate","Max Depth", "N", "Score"])
df_scores.sort_values("Score",ascending=False)

100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [02:50<00:00, 17.09s/it]


,Learning Rate,Max Depth,N,Score
110,0.900000,NaN,100,0.980885
98,0.801111,NaN,100,0.979773
86,0.702222,NaN,100,0.978662
97,0.801111,NaN,50,0.978440
107,0.801111,4.0,100,0.978217
...,...,...,...,...
15,0.108889,2.0,25,0.910202
5,0.010000,2.0,100,0.907090
6,0.010000,3.0,25,0.901978
4,0.010000,2.0,50,0.855301
